# The second case: marketing pushes back

**Week 1, Thursday. The afternoon's second case, 40 minutes, in pairs.** Four moves and eight lettered
choices, then the reply spoken aloud to a partner playing marketing. The brief is
`exercises/unguided/C2_W01_D04_pushback_STUDENT.md`.

> **The client asks.** "Your segment split is slicing the data until it says what you want. Exposed
> Retail-Plus members spent Rs 4,850 in August. That is far above the Rs 3,200 our unexposed customers
> averaged. The sale works." The marketing lead, Kalpa Retail

The table is the campaign platform's August list: 160 Retail-Plus and Retail-Core customers under the
platform's own ids, with one average August spend for each group. Replace each `__TODOn__` with the
line you choose from the comment above it; the check cell after each move compares what your choices
computed with the numbers they should reach. Run All stops at the first placeholder until you do.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

EXPOSURE = kit.load_csv("C2_W01_D04_exposure_STUDENT.csv")
CAMPAIGN = kit.load_csv("C2_W01_D04_campaigns_STUDENT.csv")[0]
PAIR = ["Retail-Plus", "Retail-Core"]


def mean(values):
    return sum(values) / len(values)


def avg(segment, flag):
    return mean([int(r["august_revenue"]) for r in EXPOSURE if r["segment"] == segment and r["exposed"] == flag])


def count(flag, segment=None):
    return sum(1 for r in EXPOSURE if r["exposed"] == flag and (segment is None or r["segment"] == segment))


print(CAMPAIGN["name"], CAMPAIGN["starts"], "to", CAMPAIGN["ends"], "at", CAMPAIGN["discount_pct"], "percent off,",
      "targeted at", CAMPAIGN["target_segment"], ";", len(EXPOSURE), "customers on the platform's list")

## Move 1. Reproduce marketing's two numbers

A pushback is answered from the same table, so first make both numbers appear.

In [ ]:
# TODO 1. Which line reproduces marketing's Rs 4,850?
#   a) avg("Retail-Plus", "no")
#   b) avg("Retail-Plus", "yes")
#   c) avg("Retail-Core", "yes")
#   d) mean([int(r["august_revenue"]) for r in EXPOSURE])
marketing_yes = __TODO1__
# TODO 2. Which line reproduces marketing's Rs 3,200?
#   a) avg("Retail-Core", "no")
#   b) avg("Retail-Plus", "no")
#   c) mean([int(r["august_revenue"]) for r in EXPOSURE if r["exposed"] == "no"])
#   d) mean([int(r["august_revenue"]) for r in EXPOSURE if r["exposed"] == "no" and r["segment"] == "Retail-Plus"])
marketing_no = __TODO2__
kit.columns(["exposed Retail-Plus", "all unexposed"], [("August spend per customer", [marketing_yes, round(marketing_no)])],
            fmt=kit.rupees, width=560, title="Marketing's comparison, reproduced")

In [ ]:
kit.check("marketing's Rs 4,850 is reproduced", marketing_yes == 4850, kit.rupees(marketing_yes))
kit.check("marketing's Rs 3,200 is reproduced", round(marketing_no) == 3200, kit.rupees(marketing_no))

## Move 2. Name the mismatch

Look at who sits in each of marketing's two groups before judging the gap between them.

In [ ]:
mix_yes = {s: count("yes", s) / count("yes") for s in PAIR}
mix_no = {s: count("no", s) / count("no") for s in PAIR}
kit.table(["Group", "Customers", "Retail-Plus", "Retail-Core"],
          [("exposed", count("yes"), count("yes", "Retail-Plus"), count("yes", "Retail-Core")),
           ("not exposed", count("no"), count("no", "Retail-Plus"), count("no", "Retail-Core"))],
          caption="Who is in each group")
kit.columns(PAIR, [("share of exposed", [round(100 * mix_yes[s]) for s in PAIR]),
                   ("share of not exposed", [round(100 * mix_no[s]) for s in PAIR])],
            fmt=lambda v: f"{v:.0f}%", width=560, title="The mix of the two groups")
# TODO 3. Marketing's Rs 3,200 averages every unexposed customer. What share of them are Retail-Core?
#   a) count("no", "Retail-Core") / len(EXPOSURE)
#   b) count("yes", "Retail-Core") / count("yes")
#   c) mix_no["Retail-Plus"]
#   d) count("no", "Retail-Core") / count("no")
core_share_unexposed = __TODO3__

In [ ]:
kit.check("three in five of marketing's unexposed customers are Retail-Core, who spend less anyway",
          round(core_share_unexposed, 2) == 0.6, f"{core_share_unexposed:.0%}")

## Move 3. Rebuild it like for like

Two fair versions: inside each segment, and the whole exposed group put into the unexposed group's
mix. Every average you quote carries the customers behind it.

In [ ]:
# TODO 4. What is the like-for-like comparison for exposed Retail-Plus?
#   a) avg("Retail-Plus", "no")
#   b) marketing_no
#   c) avg("Retail-Core", "no")
#   d) avg("Retail-Plus", "yes")
plus_like = avg("Retail-Plus", "yes") / __TODO4__ - 1
core_like = avg("Retail-Core", "yes") / avg("Retail-Core", "no") - 1
# TODO 5. Which weights put the exposed group into the unexposed group's mix?
#   a) mix_yes[s]
#   b) mix_no[s]
#   c) 0.5
#   d) count("yes", s) / len(EXPOSURE) * 2
reweighted_yes = sum(__TODO5__ * avg(s, "yes") for s in PAIR)
fair_lift = reweighted_yes / marketing_no - 1
# TODO 6. How many customers stand behind the like-for-like Retail-Plus comparison, exposed and not?
#   a) (count("yes"), count("no"))
#   b) (count("yes", "Retail-Plus"), count("no", "Retail-Core"))
#   c) (len(EXPOSURE), 0)
#   d) (count("yes", "Retail-Plus"), count("no", "Retail-Plus"))
like_counts = __TODO6__
kit.columns(PAIR, [("not exposed", [avg(s, "no") for s in PAIR]), ("exposed", [avg(s, "yes") for s in PAIR])],
            fmt=kit.rupees, width=560, title="Like for like: each segment, exposed against not exposed")
kit.bridge(("unexposed, as mixed", round(marketing_no)), [("the sale, same mix", round(reweighted_yes - marketing_no))],
           end_label="exposed, same mix", lo=2500, title="Same mix, only the sale differs (axis from Rs 2,500)")
print(f"Retail-Plus {plus_like:+.1%} on {like_counts}, Retail-Core {core_like:+.1%}, same-mix blend {fair_lift:+.1%}")

In [ ]:
kit.check("inside Retail-Plus, exposed customers spent 3.0 percent less than their unexposed peers",
          round(plus_like, 3) == -0.03, f"{plus_like:+.1%}")
kit.check("put into the unexposed group's mix, the exposed group spent 3.0 percent less", round(fair_lift, 3) == -0.03,
          f"{fair_lift:+.1%}")
kit.check("the like-for-like comparison stands on 30 exposed and 40 unexposed customers", like_counts == (30, 40),
          f"{like_counts}")

## Move 4. What the sale cost, and the offer to Marketing

The platform's list does not say whether its August spend is counted before or after the 15 percent
off. Suppose it is counted at list price, before the discount, and size what the exposed customers
actually paid Kalpa. Then size the Diwali hold-back the pair will offer, on the same list.

In [ ]:
off = int(CAMPAIGN["discount_pct"]) / 100
blend_yes = mean([int(r["august_revenue"]) for r in EXPOSURE if r["exposed"] == "yes"])
# TODO 7. At list price, how did what the exposed paid Kalpa compare with what the unexposed paid?
#   a) blend_yes * (1 - off) / marketing_no - 1
#   b) blend_yes / marketing_no - 1 - off
#   c) blend_yes / (marketing_no * (1 - off)) - 1
#   d) (blend_yes - off) / marketing_no - 1
paid_gap = __TODO7__
# TODO 8. A coin keeps one in five out of the Diwali sale inside each segment on the list. How many is that?
#   a) {s: count("yes", s) // 5 for s in PAIR}
#   b) {s: (count("yes", s) + count("no", s)) // 5 for s in PAIR}
#   c) {s: count("no", s) for s in PAIR}
#   d) {"Retail-Plus": (count("yes", "Retail-Plus") + count("no", "Retail-Plus")) // 5}
held = __TODO8__
forgone = round(sum(held[s] * avg(s, "no") * 0.06 for s in held))
kit.flow(["reproduce\nRs 4,850 and Rs 3,200", "mismatch\nwho is in each group", "like for like\n3 percent less",
          "the offer\na coin, one in five"], lit=3, title="The pair's four moves")
print(f"at list price the exposed paid {paid_gap:+.1%} against the unexposed; the hold-back {held}, "
      f"forgoing {kit.rupees(forgone)} if Marketing's 6 percent were real")

In [ ]:
kit.check("counted at list price, the exposed paid Kalpa about 9.8 percent less per customer than the unexposed",
          round(paid_gap, 3) == -0.098, f"{paid_gap:+.1%}")
kit.check("the hold-back forgoes about Rs 6,360 across both segments at Marketing's own 6 percent", forgone == 6360,
          kit.rupees(forgone))

**The reply.** Say it aloud to your partner playing marketing, in two sentences: the like-for-like
comparison with its counts, the list-price question Marketing has to answer, and the Diwali hold-back
with what it costs.

In [ ]:
kit.check_summary()
print("Post the eight letters in order, then say the reply aloud to your partner playing marketing.")